# oPool cloning — simple Run All workflow

1. Edit only the **User inputs** cell below.
2. Choose **Run All**.
3. The final cell prints progress and lists every output file.

The input can be either a two-column `name, amino-acid sequence` CSV or an existing CSV containing `name` and `dna_seq_optimized`; it is detected automatically.

## User inputs — edit this one cell only

In [ ]:
# ============================================================
# REQUIRED PROJECT SETTINGS
# ============================================================

# Amino-acid CSV or an existing *_Optimised.csv file.
INPUT_FILE = "data/AAseq_dTF001_dTF016.csv"

# Final synthesized oligo length.
OPOOL_LENGTH = 350

# Internal-overhang inventory. Use None for oPool_Optimiser/data/overhangs.csv.
OVERHANGS_FILE = None

# Four-base vector overhangs.
VECTOR_OVERHANG_1 = "TATG"
VECTOR_OVERHANG_2 = "GGAT"

# None packs automatically; set 1 for exactly one gene in every sub-pool.
GENES_PER_SUBPOOL = None

# Codon-optimization host (used only when INPUT_FILE contains amino-acid sequences).
# Built-ins: b_subtilis, c_elegans, d_melanogaster, e_coli, g_gallus,
# h_sapiens, m_musculus, m_musculus_domesticus, s_cerevisiae.
# A numeric NCBI taxonomy ID is also accepted but requires internet access.
CODON_SPECIES = "e_coli"

# ============================================================
# OPTIONAL SETTINGS — defaults are suitable for most runs
# ============================================================

OUTPUT_DIR = None       # None writes beside INPUT_FILE
RUN_NAME = None         # None uses the input file's folder name
PRIMERS_FILE = None     # None uses data/orthogonal_oligos.csv
PRIMER_MODE = "combinatorial"    # "combinatorial" or "unique_pairs"
PRIMER_START_AT = None              # used only for unique_pairs
STRIP_NTERM_MET = True
ADD_STUFFER = True
SKIP_PRIMER_ASSIGNMENT = False
TYPEIIS_RECOGNITION_SITE = "GGTCTC"
TYPEIIS_N_BASE = "A"
OVERWRITE_EXISTING = True

# Less commonly changed settings. Remove an entry to use its built-in default.
ADVANCED_OPTIONS = {
    "max_fragments": 32,
    "short_pool_max_size": 1000,
    "gc_min": 0.30,
    "gc_max": 0.70,
    "gc_window": 50,
    "random_seed": 123,
}


## Workflow — run automatically; do not edit

In [ ]:
import sys
from pathlib import Path
import pandas as pd

def find_project_root():
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        for candidate in (base, base / "oPool_Optimiser"):
            if (candidate / "scripts" / "opool_workflow.py").is_file():
                return candidate
    raise RuntimeError("Could not find oPool_Optimiser/scripts/opool_workflow.py")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

from opool_workflow import WorkflowConfig, run_workflow

config = WorkflowConfig(
    input_path=INPUT_FILE,
    output_dir=OUTPUT_DIR,
    run_name=RUN_NAME,
    overhangs_path=OVERHANGS_FILE,
    primers_path=PRIMERS_FILE,
    opool_length=OPOOL_LENGTH,
    genes_per_subpool=GENES_PER_SUBPOOL,
    codon_species=CODON_SPECIES,
    vector_oh1=VECTOR_OVERHANG_1,
    vector_oh2=VECTOR_OVERHANG_2,
    primer_mode=PRIMER_MODE,
    primer_start_at=PRIMER_START_AT,
    strip_nterm_met=STRIP_NTERM_MET,
    add_stuffer=ADD_STUFFER,
    skip_primer_assignment=SKIP_PRIMER_ASSIGNMENT,
    typeiis_site=TYPEIIS_RECOGNITION_SITE,
    typeiis_n=TYPEIIS_N_BASE,
    force=OVERWRITE_EXISTING,
    **ADVANCED_OPTIONS,
)

result = run_workflow(config)

pd.DataFrame({"Output file": [str(path) for path in result.output_files]})
